In [1]:
# Bhopal approx bounding box
from sentinelhub import BBox, CRS

bbox = BBox(
    bbox=[77.30, 23.20, 77.50, 23.35],  # [min_lon, min_lat, max_lon, max_lat]
    crs=CRS.WGS84
)

In [2]:
from sentinelhub import (
    SHConfig, SentinelHubRequest, DataCollection,
    MimeType, bbox_to_dimensions
)

# 🔐 API config
config = SHConfig()
config.sh_client_id = "ca118fee-7d8c-4806-b8f6-b92bc64ea610"
config.sh_client_secret = "ZMnmBhby8LV6d8WinTNWiWrMJ6ypAyAI"

# 📏 image size
resolution = 60  # meters
size = bbox_to_dimensions(bbox, resolution=resolution)

# 🧪 NDVI evalscript
evalscript = """
//VERSION=3
function setup() {
  return {
    input: ["B04", "B08"],
    output: { bands: 1 }
  };
}

function evaluatePixel(sample) {
  let ndvi = (sample.B08 - sample.B04) / (sample.B08 + sample.B04);
  return [ndvi];
}
"""

# 🚀 request
request = SentinelHubRequest(
    evalscript=evalscript,
    input_data=[
        SentinelHubRequest.input_data(
            data_collection=DataCollection.SENTINEL2_L2A,
            time_interval=("2024-01-01", "2024-12-31"),
        )
    ],
    responses=[
        SentinelHubRequest.output_response("default", MimeType.TIFF)
    ],
    bbox=bbox,
    size=size,
    config=config
)

# 📥 get image
ndvi_image = request.get_data()[0]

print("NDVI Shape:", ndvi_image.shape)

NDVI Shape: (283, 337)


In [3]:
import numpy as np

ndvi = ndvi_image

green = np.sum(ndvi > 0.3)
non_green = np.sum(ndvi < 0.2)

total = ndvi.size

print("Green %:", (green/total)*100)
print("Non-Green %:", (non_green/total)*100)

Green %: 88.9347914984639
Non-Green %: 11.065208501536105


In [4]:
import numpy as np

ndvi_data = ndvi  # tera NDVI array

# grid size (change kar sakta hai)
rows, cols = ndvi_data.shape
grid_size = 10

sector_h = rows // grid_size
sector_w = cols // grid_size

results = []

for i in range(grid_size):
    for j in range(grid_size):
        sector = ndvi_data[
            i*sector_h:(i+1)*sector_h,
            j*sector_w:(j+1)*sector_w
        ]
        
        avg_ndvi = np.mean(sector)
        green_ratio = np.sum(sector > 0.3) / sector.size
        
        # priority logic
        if avg_ndvi < 0.2:
            priority = "HIGH"
        elif avg_ndvi < 0.4:
            priority = "MEDIUM"
        else:
            priority = "LOW"
        
        results.append({
            "sector": f"{i},{j}",
            "avg_ndvi": round(avg_ndvi, 3),
            "green_ratio": round(green_ratio, 2),
            "priority": priority
        })

# print sample
for r in results[:5]:
    print(r)

{'sector': '0,0', 'avg_ndvi': 28.629, 'green_ratio': 0.96, 'priority': 'LOW'}
{'sector': '0,1', 'avg_ndvi': 29.084, 'green_ratio': 0.99, 'priority': 'LOW'}
{'sector': '0,2', 'avg_ndvi': 15.807, 'green_ratio': 0.84, 'priority': 'LOW'}
{'sector': '0,3', 'avg_ndvi': 9.064, 'green_ratio': 0.82, 'priority': 'LOW'}
{'sector': '0,4', 'avg_ndvi': 3.966, 'green_ratio': 0.47, 'priority': 'LOW'}


In [5]:
for r in results:
    area_factor = 100  # adjust kar sakta hai
    trees_needed = int((1 - r["green_ratio"]) * area_factor)
    
    r["trees_needed"] = trees_needed

In [6]:
import numpy as np
from PIL import Image

In [7]:
ndvi_min = np.min(ndvi)
ndvi_max = np.max(ndvi)

ndvi_norm = (ndvi - ndvi_min) / (ndvi_max - ndvi_min)

In [8]:
ndvi_img = (ndvi_norm * 255).astype(np.uint8)

In [9]:
img = Image.fromarray(ndvi_img)
img.save("bhopal_ndvi.png")

print("✅ Image saved successfully")

✅ Image saved successfully


In [10]:
import matplotlib.cm as cm

colored = cm.RdYlGn(ndvi_norm)  # apply colormap
colored_img = (colored[:, :, :3] * 255).astype(np.uint8)

Image.fromarray(colored_img).save("bhopal_ndvi_color.png")